# 01 — Descarga y Exploración de Datos Externos

## Objetivo

El dataset original de partidos (`train.csv`) contiene información básica de cada encuentro: equipos, fecha, resultado y goles. Sin embargo, para construir un modelo predictivo competitivo, necesitamos **enriquecer estos datos** con variables externas que aporten contexto sobre la calidad y recursos de cada equipo.

En este cuaderno:
1. Explicamos la **fuente de datos externa** elegida.
2. Construimos y justificamos cada **nueva variable**.
3. Integramos las variables al dataset y guardamos el resultado en `data/interim/`.

## Fuente de datos externa

Los datos externos se han obtenido de un dataset de kaggle, en concreto el siguiente:https://www.kaggle.com/datasets/xfkzujqjvx97n/football-datasets/data?select=player_performances . El dataset combina datos de **[Transfermarkt](https://www.transfermarkt.es/)**, la mayor base de datos pública sobre fútbol profesional.

Los archivos descargados se encuentran en `data/raw/external/` e incluyen:

| Archivo | Descripción | Tamaño aprox. |
|---|---|---|
| `player_market_value.csv` | Valoración de mercado histórica de cada jugador | ~24 MB |
| `player_performances.csv` | Rendimiento de jugadores por temporada y equipo | ~158 MB |
| `player_profiles.csv` | Perfiles de jugadores (edad, nacionalidad, posición) | ~26 MB |
| `player_injuries.csv` | Historial de lesiones | ~8 MB |
| `team_details.csv` | Información general de equipos | ~0.6 MB |
| `team_competitions_seasons.csv` | Participación de equipos en competiciones | ~17 MB |
| `player_teammates_played_with.csv` | Relaciones entre compañeros de equipo | ~48 MB |

### Reto principal: nombres de equipos

Los nombres de equipos en Transfermarkt son distintos a los del `train.csv`. Por ejemplo:

| `train.csv` | Transfermarkt |
|---|---|
| `Ath Bilbao` | `Athletic Bilbao` |
| `Betis` | `Real Betis Balompié` |
| `Man City` | `Manchester City` |
| `Wolves` | `Wolverhampton Wanderers` |

Para resolver esto, se ha construido un **diccionario de traducción completo** (`MAPEO_EQUIPOS` en `src/data_loader.py`) que cubre los 87 equipos presentes en el dataset.

## Setup inicial

In [2]:
# Recarga automática de módulos (útil durante el desarrollo)
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd

# Añadir la raíz del proyecto al path para poder importar src/
root_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_dir not in sys.path:
    sys.path.append(root_dir)

# Rutas a los datos
RUTA_TRAIN = '../data/raw/train.csv'
RUTA_VALUATIONS = '../data/raw/external/player_market_value.csv'
RUTA_PERFORMANCES = '../data/raw/external/player_performances.csv'
RUTA_SALIDA = '../data/interim/train_enriquecido.csv'

In [3]:
# Cargar el dataset original
df_partidos = pd.read_csv(RUTA_TRAIN)
print(f'Dataset original: {df_partidos.shape[0]} partidos, {df_partidos.shape[1]} columnas')
df_partidos.head()

Dataset original: 17480 partidos, 9 columnas


,id,division,anio_fin_temporada,fecha,local,visitante,goles_local,goles_visitante,resultado
0,10341_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Espanol,Almeria,3,3,X_empate
1,10334_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Osasuna,Girona,2,1,1_local
2,10335_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Real Madrid,Ath Bilbao,1,1,X_empate
3,10336_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Sociedad,Sevilla,2,1,1_local
4,10337_2223_ESP1,ESP_LaLiga,2023,2023-06-04,Villarreal,Ath Madrid,2,2,X_empate


---

## Variable 1: Valor de la plantilla al inicio de la temporada

### ¿Por qué es interesante?

El **valor de mercado de una plantilla** es uno de los mejores indicadores de la calidad general de un equipo. Un equipo con jugadores más valiosos suele tener mayor probabilidad de ganar, ya que el valor de mercado refleja:
- El **talento individual** de los jugadores.
- La **inversión del club** en fichajes.
- La **reputación y rendimiento reciente** de cada jugador (Transfermarkt actualiza valores según el desempeño).

### Descripción de la variable

Para cada equipo y temporada, calculamos la **suma de las valoraciones de mercado de todos los jugadores de su plantilla** al inicio de la temporada.

Se generan **dos columnas**:
- `valor_plantilla_local`: Valor total de la plantilla del equipo **local**.
- `valor_plantilla_visitante`: Valor total de la plantilla del equipo **visitante**.

### Decisiones de diseño para evitar data leakage

En un modelo predictivo, es crucial no usar información que no estaría disponible en el momento de la predicción. Para esta variable hemos tomado las siguientes precauciones:

| Decisión | Justificación |
|---|---|
| **Cutoff: 1 de agosto** | Las ligas empiezan entre el 5 y 17 de agosto. Usando solo valoraciones anteriores al 1 de agosto, garantizamos que **ningún partido de la temporada se ha jugado aún**. |
| **Última valoración conocida** | Para cada jugador, tomamos su valoración más reciente antes del cutoff, no la media ni la máxima de la temporada. Esto refleja el estado real del mercado en ese momento. |
| **Actualización de junio de Transfermarkt** | Transfermarkt publica una gran actualización de valoraciones en junio (~217.000 registros), justo al final de temporada. Esto nos da una cobertura excelente con el cutoff de agosto. |

### Limitación conocida

La composición de la plantilla (qué jugadores pertenecen a cada equipo) se obtiene del archivo `player_performances.csv`, que registra en qué equipo jugó cada jugador **durante toda la temporada**. Esto significa que incluye jugadores fichados después del inicio (por ejemplo, en el mercado de invierno de enero). Sin un dataset específico de traspasos, esta es la mejor aproximación disponible. El impacto es menor ya que la gran mayoría de jugadores de una temporada estaban en el club desde el inicio.

### Cálculo de la variable

In [4]:
from src.data_loader import preparar_valor_plantilla_inicio_temporada

# Calcular el valor de plantilla por equipo y temporada
df_valor_plantilla = preparar_valor_plantilla_inicio_temporada(RUTA_VALUATIONS, RUTA_PERFORMANCES)

print(f'Registros generados: {len(df_valor_plantilla)} (equipo × temporada)')
print(f'Equipos únicos: {df_valor_plantilla["equipo"].nunique()}')
print(f'Temporadas cubiertas: {sorted(df_valor_plantilla["season_join_key"].unique())}')
df_valor_plantilla.head(10)

ImportError: cannot import name 'preparar_valor_plantilla_inicio_temporada' from 'src.data_loader' (c:\Users\movil\Desktop\Proyecto IA futbol\Prediccion-Futbol-IA\src\data_loader.py)

### Integración con el dataset de partidos

Hacemos **dos merges**: uno para el equipo local y otro para el visitante. La clave de unión es el nombre del equipo + el año de fin de temporada.

In [ ]:
# --- Merge para el equipo LOCAL ---
df_enriquecido = pd.merge(
    df_partidos,
    df_valor_plantilla,
    left_on=['local', 'anio_fin_temporada'],
    right_on=['equipo', 'season_join_key'],
    how='left'
)
df_enriquecido = df_enriquecido.rename(columns={'valor_plantilla_inicio': 'valor_plantilla_local'})
df_enriquecido = df_enriquecido.drop(columns=['equipo', 'season_join_key'])

# --- Merge para el equipo VISITANTE ---
df_enriquecido = pd.merge(
    df_enriquecido,
    df_valor_plantilla,
    left_on=['visitante', 'anio_fin_temporada'],
    right_on=['equipo', 'season_join_key'],
    how='left'
)
df_enriquecido = df_enriquecido.rename(columns={'valor_plantilla_inicio': 'valor_plantilla_visitante'})
df_enriquecido = df_enriquecido.drop(columns=['equipo', 'season_join_key'])

print(f'Columnas del dataset enriquecido: {df_enriquecido.columns.tolist()}')
df_enriquecido.head()

### Validación de la variable

In [ ]:
# Cobertura: ¿cuántos partidos tienen valor de plantilla?
total = len(df_enriquecido)
cob_local = df_enriquecido['valor_plantilla_local'].notna().sum()
cob_visit = df_enriquecido['valor_plantilla_visitante'].notna().sum()
cob_ambos = (df_enriquecido['valor_plantilla_local'].notna() & df_enriquecido['valor_plantilla_visitante'].notna()).sum()

print(f'Cobertura equipo local:     {cob_local:>5}/{total} ({100*cob_local/total:.1f}%)')
print(f'Cobertura equipo visitante: {cob_visit:>5}/{total} ({100*cob_visit/total:.1f}%)')
print(f'Cobertura ambos equipos:    {cob_ambos:>5}/{total} ({100*cob_ambos/total:.1f}%)')
print(f'\nNota: Los partidos sin cobertura corresponden a temporadas antiguas sin datos en Transfermarkt.')

In [ ]:
# Top 10 equipos con mayor valor de plantilla (temporada 2022/23)
top_equipos = (
    df_enriquecido[df_enriquecido['anio_fin_temporada'] == 2023]
    [['local', 'valor_plantilla_local']]
    .drop_duplicates('local')
    .sort_values('valor_plantilla_local', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top_equipos['valor_plantilla_local'] = (top_equipos['valor_plantilla_local'] / 1e6).round(0).astype(int).astype(str) + 'M €'
top_equipos.columns = ['Equipo', 'Valor plantilla']
print('Top 10 plantillas más valiosas (inicio temporada 2022/23):')
top_equipos

In [ ]:
# Estadísticas descriptivas del valor de plantilla
print('Estadísticas descriptivas (en millones de €):')
print(
    df_enriquecido[['valor_plantilla_local', 'valor_plantilla_visitante']]
    .describe()
    .apply(lambda x: (x / 1e6).round(1))
)

---

## Guardado del dataset enriquecido

Guardamos el dataset con las nuevas variables en `data/interim/train_enriquecido.csv`. Este archivo será el punto de partida para los siguientes cuadernos de análisis y modelado.

**Nuevas columnas añadidas:**
- `valor_plantilla_local`: Valor de mercado de la plantilla del equipo local al inicio de la temporada (en €).
- `valor_plantilla_visitante`: Valor de mercado de la plantilla del equipo visitante al inicio de la temporada (en €).

In [ ]:
# Guardar el dataset enriquecido
df_enriquecido.to_csv(RUTA_SALIDA, index=False)

print(f'Dataset guardado en: {RUTA_SALIDA}')
print(f'Dimensiones: {df_enriquecido.shape[0]} filas × {df_enriquecido.shape[1]} columnas')
print(f'\nColumnas finales:')
for i, col in enumerate(df_enriquecido.columns, 1):
    nuevo = ' ← NUEVA' if 'valor_plantilla' in col else ''
    print(f'  {i}. {col}{nuevo}')